# Strategy comparison

Compare registered monthly allocation strategies on one interactive chart. This notebook defaults to the real Shiller dataset; synthetic data is an explicit smoke-test option only.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from retail_sp500.backtest import BacktestConfig, run_many
from retail_sp500.data import enrich_with_risk_free_rate, load_shiller_data, synthetic_market_data
from retail_sp500.plotting import comparison_figure
from retail_sp500.strategies import select_strategies

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Load data

`USE_SYNTHETIC = False` downloads real monthly Shiller data. Turn it on only for an offline smoke test; synthetic dates and values are not market history.

In [ ]:
USE_SYNTHETIC = False
INCLUDE_FRED = False

market = synthetic_market_data(periods=600) if USE_SYNTHETIC else load_shiller_data()
if INCLUDE_FRED and not USE_SYNTHETIC:
    market = enrich_with_risk_free_rate(market)

print({
    "source": "synthetic smoke test" if USE_SYNTHETIC else "real Shiller monthly data",
    "start": market.index.min(),
    "end": market.index.max(),
    "rows": len(market),
})
assert USE_SYNTHETIC or market.index.max() <= pd.Timestamp.today().normalize()
market.tail()

## Select strategies

Edit the keys below. Unsupported strategies are reported instead of receiving fabricated inputs.

In [ ]:
strategy_keys = [
    "buy-hold",
    "trend-10m",
    "vol-target-12",
    "trend-vol-12",
    "fractional-kelly",
]

strategies, skipped = select_strategies(
    strategy_keys,
    market,
    skip_unavailable=True,
)

skipped

In [ ]:
config = BacktestConfig(
    initial_cash=100_000,
    monthly_contribution=1_000,
    cash_annual_return=0.0,
)

results = run_many(market, strategies, config)

## Metrics and graph

In [ ]:
metrics = pd.DataFrame(
    {name: result.metrics for name, result in results.items()}
).T

metrics.sort_values("ending_value", ascending=False)

In [ ]:
comparison_figure(results).show()

## Inspect one strategy

In [ ]:
selected_name = next(iter(results))
results[selected_name].history.tail(24)